# LA Studio TTS — VoxCPM2

This notebook loads exactly `voxcpm2` (`openbmb/VoxCPM2`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's TTS panel.


In [ ]:
!nvidia-smi
!git clone --quiet https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM
!git -C /content/VoxCPM checkout --quiet 616d3d3e630a
%pip install -q -e /content/VoxCPM "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_tts_worker.py')
WORKER.write_text('import io\nimport os\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_TTS_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass SpeechRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice: str = Field(default="auto", max_length=160)\n    language: str = Field(default="auto", max_length=40)\n    speed: float = Field(default=1.0, ge=0.25, le=4.0)\n    response_format: str = "wav"\n    settings: dict = Field(default_factory=dict)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef wav_response(samples, sample_rate: int):\n    audio = np.asarray(samples, dtype=np.float32).reshape(-1)\n    if audio.size == 0:\n        raise RuntimeError("the selected model returned no audio")\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    if not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned non-finite audio")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\nfrom voxcpm import VoxCPM\n\nMODEL_ID = "voxcpm2"\nMODEL_NAME = "VoxCPM2"\nUPSTREAM_MODEL = "openbmb/VoxCPM2"\nSUPPORTED_LANGUAGES = ["auto", "vi", "en", "zh", "ja", "ko", "fr", "de", "es", "it", "pt", "th"]\nSUPPORTED_VOICES = ["auto"]\nMODEL = VoxCPM.from_pretrained(UPSTREAM_MODEL, load_denoiser=False, optimize=True, device="cuda")\nif "cuda" not in str(MODEL.model.device).lower():\n    raise RuntimeError("VoxCPM2 did not load on CUDA")\n\ndef synthesize_exact_model(request: SpeechRequest):\n    audio = MODEL.generate(\n        text=request.input,\n        cfg_value=float(request.settings.get("cfg_value", 2.0)),\n        inference_timesteps=int(request.settings.get("inference_timesteps", 10)),\n        normalize=bool(request.settings.get("normalize", True)),\n        seed=request.settings.get("seed"),\n    )\n    return audio, 48000\n\napp = FastAPI(title=f"LA Studio TTS — {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "variant": "fixed",\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "tts",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "variant": "fixed",\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "voices": SUPPORTED_VOICES,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/speech")\ndef speech(request: SpeechRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if request.response_format.strip().lower() != "wav":\n        raise HTTPException(status_code=422, detail="this worker returns WAV audio only")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab TTS worker is busy; retry shortly")\n    try:\n        samples, sample_rate = synthesize_exact_model(request)\n        return wav_response(samples, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} synthesis failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
# LA Studio worker launch contract: launch-2026-07-30.1
import json
import os
import queue
import re
import secrets
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CAPABILITY_LABEL = 'TTS'
MODEL_ID = 'voxcpm2'
PORT = 3921
TOKEN_ENV = 'LA_STUDIO_COLAB_TTS_TOKEN'
URL_ENV = 'LA_STUDIO_COLAB_TTS_URL'
MODEL_ENV = 'LA_STUDIO_COLAB_TTS_MODEL'
WORKER_LOG = Path('/content/la_studio_tts_worker.log')
STARTUP_TIMEOUT_SECONDS = 20 * 60
TUNNEL_TIMEOUT_SECONDS = 90
TOKEN = secrets.token_urlsafe(32)


def port_is_occupied(port: int) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.5):
            return True
    except OSError:
        return False


def worker_log_tail() -> str:
    try:
        return WORKER_LOG.read_text(encoding="utf-8", errors="replace")[-12000:]
    except FileNotFoundError:
        return "(worker log was not created)"


def stop_process(process) -> None:
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()


if port_is_occupied(PORT):
    raise RuntimeError(
        f"Port {PORT} is already occupied by an earlier Colab worker. "
        "Use Runtime > Disconnect and delete runtime, then Run all once for this exact model."
    )

env = os.environ.copy()
env[TOKEN_ENV] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
worker = None
tunnel = None

with WORKER_LOG.open("w", encoding="utf-8", buffering=1) as worker_output:
    worker = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", 'la_studio_tts_worker:app', "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content",
        env=env,
        stdout=worker_output,
        stderr=subprocess.STDOUT,
    )
    print(f"Starting exact CUDA {CAPABILITY_LABEL} worker; initial model load can take several minutes.")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SECONDS
    last_error = "worker has not answered /health yet"
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        exit_code = worker.poll()
        if exit_code is not None:
            raise RuntimeError(
                f"The exact-model {CAPABILITY_LABEL} worker exited before becoming ready (exit code {exit_code}).\n\n"
                "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
            )
        try:
            request = urllib.request.Request(
                f"http://127.0.0.1:{PORT}/health",
                headers={"Authorization": "Bearer " + TOKEN},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                health = json.loads(response.read().decode("utf-8"))
            if (response.status == 200
                    and health.get("ready") is True
                    and str(health.get("device", "")).lower() == "cuda"
                    and str(health.get("model", "")).strip().lower() == MODEL_ID
                    and health.get("cpu_fallback") is False):
                print("Exact CUDA worker is ready:", health)
                break
            last_error = "unexpected /health response: " + json.dumps(health, ensure_ascii=False)
        except urllib.error.HTTPError as error:
            last_error = f"/health returned HTTP {error.code}: " + error.read().decode("utf-8", errors="replace")[:1000]
        except Exception as error:
            last_error = f"/health is not ready: {type(error).__name__}: {error}"
        if time.monotonic() >= next_report:
            print("Waiting for the exact CUDA model…", last_error)
            next_report = time.monotonic() + 30
        time.sleep(2)
    else:
        stop_process(worker)
        raise RuntimeError(
            f"The exact-model {CAPABILITY_LABEL} worker did not become CUDA-ready within "
            f"{STARTUP_TIMEOUT_SECONDS // 60} minutes. Last health-check result: {last_error}\n\n"
            "---- LA Studio worker log (last 12,000 characters) ----\n" + worker_log_tail()
        )


def cloudflared_ready() -> bool:
    try:
        return subprocess.run(
            ["cloudflared", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    except OSError:
        return False


def ensure_cloudflared() -> None:
    if cloudflared_ready():
        return
    package_path = "/content/la-studio-cloudflared.deb"
    download = subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "4", "--retry-all-errors",
            "--output", package_path,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if download.returncode != 0:
        detail = download.stdout[-1200:].strip() or "no download output"
        raise RuntimeError("Could not download cloudflared: " + detail)
    install = subprocess.run(
        ["dpkg", "-i", package_path], text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False,
    )
    if install.returncode != 0 or not cloudflared_ready():
        detail = install.stdout[-1200:].strip() or "no installation output"
        raise RuntimeError("Could not install cloudflared: " + detail)


ensure_cloudflared()
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()


def collect_tunnel_output() -> None:
    assert tunnel.stdout is not None
    for line in tunnel.stdout:
        tunnel_lines.put(line)


threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = ""
recent_tunnel_lines = []
deadline = time.monotonic() + TUNNEL_TIMEOUT_SECONDS
while time.monotonic() < deadline and not public_url:
    if tunnel.poll() is not None:
        break
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_lines.append(line.rstrip())
    recent_tunnel_lines = recent_tunnel_lines[-10:]
    print(line, end="")
    match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
    if match:
        # The desktop Check Colab action is the authoritative public endpoint,
        # bearer-token, capability, and exact-model verification.
        public_url = match.group(0)

if not public_url:
    stop_process(tunnel)
    stop_process(worker)
    tail = "\n".join(recent_tunnel_lines) or "(no cloudflared output)"
    raise RuntimeError(
        f"cloudflared did not publish a trycloudflare URL within {TUNNEL_TIMEOUT_SECONDS} seconds.\n"
        "---- cloudflared output ----\n" + tail
    )

os.environ[URL_ENV] = public_url
os.environ[TOKEN_ENV] = TOKEN
os.environ[MODEL_ENV] = MODEL_ID
print("\nLA Studio exact-model Colab worker is ready")
print(URL_ENV + "=" + public_url)
print(TOKEN_ENV + "=" + TOKEN)
print(MODEL_ENV + "=" + MODEL_ID)
print("Click Check Colab in the matching LA Studio feature before running it.")
